# World

> Run a policy in a pool of envs, to collect datasets or evaluate it (stable-worldmodel API).

In [ ]:
#| default_exp world.world

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
from __future__ import annotations

import contextlib
from collections import defaultdict
from functools import partial
from pathlib import Path
from typing import Any, Callable, Iterator

import numpy as np

from stable_marl.envs.base import MultiAgentEnv
from stable_marl.envs.registry import make
from stable_marl.policy import BasePolicy
from stable_marl.world.env_pool import EnvPool, episode_done

RESET_MODES = ('auto', 'wait')


def _make_env(env_name: str, wrappers: list, **kwargs) -> MultiAgentEnv:
    env = make(env_name, **kwargs)
    for wrapper in wrappers:
        env = wrapper(env)
    return env

In [ ]:
#| export
class World:
    """
    Drives a policy through a pool of ``num_envs`` copies of a registered environment::

        world = World('MultiGrid-FindGoal-15x15-v0', num_envs=8, agents=2)
        world.set_policy(RandomPolicy(seed=0))
        world.collect('data/findgoal.h5', episodes=500, seed=0)    # record a dataset
        results = world.evaluate(episodes=100, seed=10_000)        # evaluate the policy

    Same interface as stable-worldmodel's ``World``. One difference in behaviour: episodes are
    assigned to the envs in advance (env ``i`` plays episodes ``i, i + num_envs, ...``) and
    episode ``k`` is reset with seed ``seed + k``, so results only depend on `seed` and the
    policy, not on `num_envs` or on which env finishes first.

    Attributes populated during a run:

    * ``infos``: stacked info dict of the last reset / step (see :class:`EnvPool`), plus
      ``_needs_flush`` (num_envs,): the envs that just started an episode
    * ``rewards``, ``terminateds``, ``truncateds``: (num_envs, num_agents) outputs of the last step

    Parameters
    ----------
    env_name : str
        Id of a registered environment (see ``stable_marl.list_envs()``)
    num_envs : int
        Number of envs in the pool
    max_episode_steps : int, optional
        Step limit of an episode (the env's ``max_steps``; default: the env's own)
    add_pixels : bool
        Add a render of the whole env as ``pixels`` to the infos
    pre_wrappers : list of callables
        Wrappers applied to each env (e.g. ``FullyObsWrapper``)
    **kwargs
        Passed to the env (e.g. ``agents=2``, ``tile_size=16``)
    """
    def __init__(
        self,
        env_name: str,
        num_envs: int = 1,
        max_episode_steps: int | None = None,
        add_pixels: bool = False,
        pre_wrappers: list[Callable] | None = None,
        **kwargs: Any):
        if add_pixels:
            kwargs.setdefault('render_mode', 'rgb_array')
        if max_episode_steps is not None:
            kwargs['max_steps'] = max_episode_steps
        self.env_name, self.env_kwargs = env_name, kwargs
        env_fn = partial(_make_env, env_name, list(pre_wrappers or []), **kwargs)
        self.envs = EnvPool([env_fn] * num_envs, add_pixels=add_pixels)
        self.policy: BasePolicy | None = None
        self.infos: dict = {}
        self.rewards = self.terminateds = self.truncateds = None

    @property
    def num_envs(self) -> int:
        return self.envs.num_envs

    @property
    def num_agents(self) -> int:
        return self.envs.num_agents

    @property
    def metadata(self) -> dict[str, Any]:
        "Where a collected dataset comes from (stored with it)."
        return {'env_name': self.env_name, 'env_kwargs': self.env_kwargs, 'num_agents': self.num_agents}

    def close(self):
        self.envs.close()

    def set_policy(self, policy: BasePolicy):
        "Attach a policy, and apply its ``seed`` if it has one."
        self.policy = policy
        policy.set_env(self.envs)
        if hasattr(policy, '_set_seed') and getattr(policy, 'seed', None) is not None:
            policy._set_seed(policy.seed)

    def reset(self, seed=None, options=None):
        "Reset every env and refresh ``infos``."
        _, self.infos = self.envs.reset(seed=seed, options=options)
        self.infos['_needs_flush'] = np.ones(self.num_envs, bool)
        shape = (self.num_envs, self.num_agents)
        self.rewards, self.terminateds, self.truncateds = np.zeros(shape, np.float32), np.zeros(shape, bool), np.zeros(shape, bool)

In [ ]:
#| export
from fastcore.utils import patch

@patch
def _run_iter(
    self: World,
    episodes: int | None = None,
    seed: int | None = None,
    options: dict | None = None,
    mode: str = 'auto',
    on_step: Callable[[World, np.ndarray], None] | None = None) -> Iterator[tuple[int, int]]:
    """
    Play `episodes` episodes with the policy, yielding ``(env_index, episode_index)`` when one
    ends, before that env is reset (so its final state can still be read).

    `mode` 'auto' resets finished envs until `episodes` are played; 'wait' plays one episode per
    env (at most `episodes`) and leaves finished envs idle. ``on_step(world, mask)`` is called
    after every reset and step, `mask` marking the envs it concerns.
    """
    assert mode in RESET_MODES, f"reset_mode must be one of {RESET_MODES}"
    if self.policy is None:
        raise RuntimeError("No policy set: call `set_policy` first")
    E = self.num_envs
    episodes = E if episodes is None else episodes
    queues = [list(range(i, episodes, E))[:1 if mode == 'wait' else None] for i in range(E)]
    current = np.array([q.pop(0) if q else -1 for q in queues])
    active = current >= 0
    seed_of = lambda k: None if seed is None or k < 0 else seed + int(k)

    self.reset(seed=[seed_of(k) for k in current], options=options)
    if on_step: on_step(self, active)

    while active.any():
        actions = self.policy.get_action(self.infos)
        _, self.rewards, self.terminateds, self.truncateds, self.infos = self.envs.step(actions, mask=active)
        self.infos['_needs_flush'] = np.zeros(E, bool)
        if on_step: on_step(self, active)

        done = active & episode_done(self.terminateds, self.truncateds)
        for i in np.flatnonzero(done):
            yield int(i), int(current[i])
        if not done.any():
            continue

        restart = done & np.array([bool(q) for q in queues])
        current[done & ~restart], active[done & ~restart] = -1, False
        for i in np.flatnonzero(restart):
            current[i] = queues[i].pop(0)
        if restart.any():
            _, self.infos = self.envs.reset(seed=[seed_of(k) for k in current], options=options, mask=restart)
            self.terminateds[restart], self.truncateds[restart] = False, False
            self.infos['_needs_flush'] = restart.copy()
            if on_step: on_step(self, restart)

In [ ]:
#| export
@patch
def evaluate(
    self: World,
    episodes: int | None = None,
    seed: int | None = None,
    options: dict | None = None,
    reset_mode: str | None = None) -> dict[str, Any]:
    """
    Run the policy and return, per episode (indexed by episode):

    * ``episode_successes``: whether the episode ended by termination (every agent terminated)
      rather than by the step limit
    * ``episode_returns`` (episodes, num_agents): sum of each agent's rewards
    * ``episode_lengths``: number of steps
    * ``seeds``: reset seed (-1 if unseeded)

    and ``success_rate`` (in percent, as in stable-worldmodel), ``mean_return`` (summed over
    agents) and ``mean_length``. `reset_mode` 'wait' plays one episode per env.
    """
    mode = reset_mode or 'auto'
    E, A = self.num_envs, self.num_agents
    n = E if episodes is None else (episodes if mode == 'auto' else min(episodes, E))
    results = {
        'success_rate': 0.0,
        'episode_successes': np.zeros(n, bool),
        'episode_returns': np.zeros((n, A)),
        'episode_lengths': np.zeros(n, np.int64),
        'seeds': np.full(n, -1, np.int64),
    }
    returns = np.zeros((E, A))

    def on_step(world, mask):
        for i in np.flatnonzero(mask):
            returns[i] = 0 if world.infos['_needs_flush'][i] else returns[i] + world.infos['reward'][i, 0]

    for i, k in self._run_iter(n, seed=seed, options=options, mode=mode, on_step=on_step):
        results['episode_successes'][k] = self.terminateds[i].all()
        results['episode_returns'][k] = returns[i]
        results['episode_lengths'][k] = self.infos['step_idx'][i, 0]
        results['seeds'][k] = self.envs.seeds[i]

    if n:
        results['success_rate'] = float(results['episode_successes'].mean()) * 100.0
        results['mean_return'] = float(results['episode_returns'].sum(axis=1).mean())
        results['mean_length'] = float(results['episode_lengths'].mean())
    return results

In [ ]:
#| export
@patch
def collect(
    self: World,
    path: str | Path | None = None,
    episodes: int = 0,
    seed: int | None = None,
    options: dict | None = None,
    format: str = 'hdf5',
    writer: Any = None,
    progress: bool = True):
    """
    Play `episodes` episodes and write them to `path` in the dataset `format` (or to a
    ready-made `writer`, e.g. a writer opened with ``mode='overwrite'``).

    Every info key becomes a column with one row per step, from the reset to the last step,
    as in stable-worldmodel: row ``t`` holds the infos at step ``t``, whose ``reward``,
    ``terminated`` and ``truncated`` come from the step that led there (``nan`` / False on
    the first row), while ``action`` is the action taken at step ``t`` (``nan`` on the last row).
    Keys starting with ``_`` are skipped.
    """
    from stable_marl.data import get_format

    if (path is None) == (writer is None):
        raise ValueError("World.collect: pass exactly one of `path` or `writer`")
    writer = get_format(format).open_writer(path) if writer is None else writer
    if hasattr(writer, 'metadata'):
        writer.metadata.update(self.metadata)
    buffers = [defaultdict(list) for _ in range(self.num_envs)]

    def on_step(world, mask):
        for col, data in world.infos.items():
            if col.startswith('_') or not isinstance(data, np.ndarray):
                continue
            for i in np.flatnonzero(mask):
                buffers[i][col].append(data[i, 0].copy())

    bar = _progress_bar(episodes, 'Recording') if progress else contextlib.nullcontext()
    with writer as w, bar as pbar:
        def episode_iter():
            for i, _ in self._run_iter(episodes, seed=seed, options=options, mode='auto', on_step=on_step):
                episode = {col: np.stack(rows) for col, rows in buffers[i].items()}
                buffers[i].clear()
                if 'action' in episode:   # action taken at each step: the reset's nan moves last
                    episode['action'] = np.roll(episode['action'], -1, axis=0)
                if pbar is not None: pbar.update(1)
                yield episode
        w.write_episodes(episode_iter())


def _progress_bar(total: int, desc: str):
    try:
        from tqdm import tqdm
    except ImportError:
        return contextlib.nullcontext()
    return tqdm(total=total, desc=desc)

### Tests

In [ ]:
import tempfile, torch
from stable_marl.policy import RandomPolicy, FeedForwardPolicy
from stable_marl.envs.multigrid import GoToGoalPolicy
from stable_marl.data import HDF5Dataset

world = World('MultiGrid-Empty-6x6-v0', num_envs=3, agents=2)
world.set_policy(RandomPolicy(seed=0))
res = world.evaluate(episodes=7, seed=100)
assert list(res['seeds']) == list(range(100, 107)) and res['episode_returns'].shape == (7, 2)
assert len(world.evaluate(seed=0, reset_mode='wait')['seeds']) == 3

# the expert reaches the goal, and results do not depend on the number of envs
for num_envs in (1, 3):
    world = World('MultiGrid-Empty-6x6-v0', num_envs=num_envs, agents=2)
    world.set_policy(GoToGoalPolicy())
    res = world.evaluate(episodes=5, seed=0)
    assert res['success_rate'] == 100.0, res
    if num_envs == 1: ref = res
    else: assert np.array_equal(ref['episode_lengths'], res['episode_lengths'])

# a torch model as policy
class Turn(torch.nn.Module):
    def get_action(self, info):
        return torch.zeros(info['direction'].shape[0], info['direction'].shape[2], dtype=torch.long)
world.set_policy(FeedForwardPolicy(Turn()))
assert world.evaluate(episodes=2, seed=0)['success_rate'] == 0.0

# collect -> hdf5 dataset
with tempfile.TemporaryDirectory() as tmp:
    world = World('MultiGrid-Empty-6x6-v0', num_envs=2, agents=2, max_episode_steps=10)
    world.set_policy(RandomPolicy(seed=1))
    world.collect(f'{tmp}/empty.h5', episodes=3, seed=5, progress=False)
    ds = HDF5Dataset(path=f'{tmp}/empty.h5', num_steps=4)
    assert len(ds.lengths) == 3 and ds.metadata['env_name'] == 'MultiGrid-Empty-6x6-v0'
    item = ds[0]
    assert item['pov'].shape == (4, 2, 224, 224, 3) and item['action'].shape == (4, 2)
    assert {'image', 'state', 'reward', 'terminated', 'truncated', 'step_idx', 'id', 'seed'} <= set(ds.column_names)

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()